In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: SETUP — Clone Repo & Install Packages
# ═══════════════════════════════════════════════════════════════

import os
import sys

# Clone repo
!git clone https://github.com/aneek22112007-tech/SocraticRL
%cd SocraticRL

# Add to path IMMEDIATELY
sys.path.insert(0, '/content/SocraticRL')

print("✅ Repo cloned and path set")

# Install packages
!pip install -q transformers torch peft wandb datasets accelerate

print("✅ Packages installed")

# Login to HuggingFace
from huggingface_hub import login
login()

print("✅ HuggingFace login complete")

# Login to W&B
import wandb
wandb.login()

print("\n🎉 Setup complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Load Model (Qwen2.5-7B with LoRA)
# ═══════════════════════════════════════════════════════════════

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Loading Qwen2.5-7B...")

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
tokenizer.pad_token = tokenizer.eos_token

print("✅ Model loaded")

# Add LoRA adapters
from peft import get_peft_model, LoraConfig, TaskType

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, peft_config)
print("✅ LoRA adapters added")
print(f"Trainable params: {model.print_trainable_parameters()}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Sanity Check — Verify Reward Function
# ═══════════════════════════════════════════════════════════════

from reward import compute_reward

print("Testing reward function...\n")

good = 'If you dropped a feather and a hammer in a vacuum, what would you expect to happen?'
bad  = 'The answer is that heavier objects fall faster due to gravity.'

progress_keywords = ['acceleration', 'same rate', 'air resistance', 'galileo', 'mass', 'vacuum']

r_good = compute_reward(good, 'falling objects', progress_keywords, understanding_score=0.1, turn_count=1, question_history=[])
r_bad  = compute_reward(bad,  'falling objects', progress_keywords, understanding_score=0.1, turn_count=1, question_history=[])

print(f'Good question reward: {r_good.reward:+.2f}  ({r_good.feedback})')
print(f'Bad  answer  reward:  {r_bad.reward:+.2f}  ({r_bad.feedback})')

assert r_good.reward > 0, 'Good question should get positive reward'
assert r_bad.reward  < 0, 'Direct answer should get negative reward'

print('\n✅ Reward function OK')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Build Training Dataset
# ═══════════════════════════════════════════════════════════════

from datasets import Dataset
from students.scenarios_expanded import TRAINING_SCENARIOS_EXPANDED

print(f"Building dataset from {len(TRAINING_SCENARIOS_EXPANDED)} scenarios...")

rows = []

for scenario in TRAINING_SCENARIOS_EXPANDED:
    text = f"""You are a Socratic tutor. Student believes: {scenario.misconception}. Topic: {scenario.topic}. Ask a Socratic question:"""
    rows.append({"text": text})

dataset = Dataset.from_list(rows)
print(f"✅ Dataset size: {len(dataset)} prompts")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: Initialize W&B
# ═══════════════════════════════════════════════════════════════

import wandb

wandb.init(project='socratic-rl-training', name='simple-training-run')
print(f"✅ W&B initialized: {wandb.run.url}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: TRAIN — Simple Supervised Fine-Tuning
# ═══════════════════════════════════════════════════════════════

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

print("Preparing training...")

# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"✅ Dataset tokenized: {len(tokenized_dataset)} samples")

# Training arguments
training_args = TrainingArguments(
    output_dir="./socratic-agent",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=50,
    report_to="wandb",
    fp16=False,
    bf16=False,
)

# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("\n🚀 Starting training...")
print(f"Dataset size: {len(tokenized_dataset)} samples")
print(f"This will take ~15-20 minutes\n")

trainer.train()

print("\n✅ Training complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: Save Model & Push to Hub
# ═══════════════════════════════════════════════════════════════

print("Saving model...")
model.save_pretrained('socratic-agent-final')
tokenizer.save_pretrained('socratic-agent-final')
print("✅ Saved to ./socratic-agent-final")

HF_USERNAME = 'aneek22112007-tech'
repo_id = f'{HF_USERNAME}/socratic-rl-agent'

print(f"\nPushing to HuggingFace Hub: {repo_id}")
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)
print(f"✅ Pushed to https://huggingface.co/{repo_id}")

print(f"\n📊 W&B Dashboard: {wandb.run.url}")
print("\n✅ All done!")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: Download Training Loss Curve from W&B
# ═══════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import wandb

print("Downloading W&B data...")

run = wandb.run
history = run.history()

if 'loss' in history.columns:
    steps = history['step'].values
    losses = history['loss'].values
    
    print(f"Loss stats: Initial={losses[0]:.2f}, Final={losses[-1]:.2f}")
    
    plt.figure(figsize=(10, 6))
    plt.plot(steps, losses, linewidth=2.5, color='#1D9E75')
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.title('SocraticRL — Training Loss')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('training_loss.png', dpi=150)
    print("✅ Saved training_loss.png")
    plt.show()

from google.colab import files
files.download('training_loss.png')
print(f"\n📊 W&B Run: {run.url}")
